# 02 — Split dosyaları (Faz 1)

**Bilimsel amaç:** Hangi katılımcının hangi turda eğitimde, doğrulamada ve testte olacağını **hiçbir model sonucuna bakmadan**, bir kez ve sabit olarak belirlemek (D-003, D-009, D-024, D-029). Bütün deneyler bu dosyaları okur. Böylece:
- aynı kişinin sesi hem eğitimde hem testte olamaz;
- bütün modeller aynı katılımcılarda karşılaştırılır.

**Bu notebook model EĞİTMEZ.** GPU gerekmez: *Runtime → Change runtime type → CPU*.

**Ön koşul:** `01_data_audit.ipynb` çalıştırılmış olmalı. Drive `data_derived/` içinde şunlar bulunmalı:
- `participants.csv`
- `audio_inventory.csv`
- `recording_map.csv`

**Çıktılar:**
- Drive `data_derived/participant_context.csv`. Yoksa bu notebook üretir.
- Drive `data_derived/splits/outer_r0.csv … outer_r4.csv`. Katılımcı düzeyi veri → git'e girmez (D-014).
- Kontrol: üretilen dosyaların sha256'ı, repo'daki `reports/splits/splits_manifest.json` ile **aynı** olmalı.

## 0. Oturum kurulumu

In [ ]:
import sys, platform
import numpy, pandas, sklearn
print("Python:", sys.version.split()[0], "|", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
SPLITS_DIR  = DERIVED / 'splits'
GITHUB_USER = 'alperarslan19'
REPO_NAME   = 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME

for name in ['participants.csv', 'audio_inventory.csv', 'recording_map.csv']:
    assert (DERIVED / name).exists(), f'Bulunamadı: {DERIVED / name} -> önce 01_data_audit.ipynb'
print('Girdiler mevcut.')

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız — token / kullanıcı adı / repo adı?'
print('Repo commit:', subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt

## 1. Script testi (sahte veriyle, ~10 s)
Gerçek veriye dokunmadan önce, split kurallarının (sızıntı yok, belirlenimci, seyrek tabaka birleştirme, hata yakalama) çalıştığını doğrular. `Tüm kontroller geçti` görmeden devam etme.

In [ ]:
!cd {REPO_DIR} && python tests/test_make_splits.py

## 2. Katılımcı bağlam tablosu (`participant_context.csv`)
`make_splits.py`, dönemi D-024'e göre **dosya zaman damgasından gelen kayıt tarihiyle** hesaplar. Bu tarih `participant_context.csv` içindedir. Dosya Drive'da yoksa `analyze_recording_context.py` ile üretilir. Raporu repo'ya değil geçici klasöre yazılır, çünkü repo'daki rapor zaten var.

In [ ]:
CTX = DERIVED / 'participant_context.csv'
if not CTX.exists():
    !cd {REPO_DIR} && python scripts/analyze_recording_context.py --participants {DERIVED}/participants.csv --inventory {DERIVED}/audio_inventory.csv --out-private {DERIVED} --out-report /content/recording_context_check
assert CTX.exists(), 'participant_context.csv üretilemedi'
print('Hazır:', CTX)

## 3. Split dosyalarını üret
Manifest repo'ya değil geçici klasöre yazılır. Repo'daki manifest referanstır; ikisi aşağıda karşılaştırılır.

In [ ]:
!cd {REPO_DIR} && python scripts/make_splits.py --context {CTX} --recording-map {DERIVED}/recording_map.csv --out-dir {SPLITS_DIR} --report-dir /content/splits_report

## 4. Yeniden üretilebilirlik kontrolü
Aynı girdi + aynı kurallar → aynı dosya. sha256'lar repo manifestiyle aynıysa split, Claude'un ürettiğiyle **birebir aynı** demektir. Farklıysa dosyaları **kullanma**; olası nedenler:
- `scikit-learn` sürümü farklı;
- `participant_context.csv` farklı (ör. farklı `audio_inventory.csv`).

Sürüm bilgisi iki manifestte de yazıyor.

In [ ]:
import json, hashlib
ref = json.loads((REPO_DIR / 'reports/splits/splits_manifest.json').read_text())
new = json.loads(Path('/content/splits_report/splits_manifest.json').read_text())
ok = True
for name, h in ref['split_files_sha256'].items():
    same = new['split_files_sha256'].get(name) == h
    ok &= same
    print(f"{name}: {'AYNI ✓' if same else 'FARKLI ✗'}")
print('Kohort referans:', ref['cohort']['n'], ref['cohort']['asthma'], ref['cohort']['healthy'],
      '| yeni:', new['cohort']['n'], new['cohort']['asthma'], new['cohort']['healthy'])
if not ok:
    print('Sürümler — referans:', ref['versions'], '\n           yeni:', new['versions'])
assert ok, 'Split dosyaları referansla aynı değil — kullanmadan önce nedenini bul (yukarıdaki açıklama).'

## 5. Bir fold'a bakış (nasıl kullanılır?)
Her dosyada her dış fold `k` için tüm katılımcılar bir kez listelenir:
- `role == 'test'` → o fold'un test kümesi;
- `is_inner_val` → early stopping / eşik için iç doğrulama;
- geri kalan `train` → eğitim.

In [ ]:
import pandas as pd
s = pd.read_csv(SPLITS_DIR / 'outer_r0.csv')
p = pd.read_csv(CTX)[['participant_id', 'label']]
blk = s[s.outer_fold == 0].merge(p, on='participant_id')
blk['küme'] = blk.role.where(~blk.is_inner_val, 'inner_val')
print(pd.crosstab(blk['küme'], blk['label'].map({1: 'astım', 0: 'sağlıklı'}), margins=True))